In [8]:
%pip install prefect

Note: you may need to restart the kernel to use updated packages.


In [9]:
# hacer docker compose up dentro de esta carpeta

In [1]:
from prefect import flow, task
from prefect.logging import get_run_logger
from pathlib import Path

import polars as pl

In [21]:
DATA_PATH = Path("../data_raw")

# Clientes

In [63]:
@task
def leer_clientes():

    logger = get_run_logger()

    logger.info("Leyendo clientes...")

    clientes = pl.read_csv(
        DATA_PATH / "clientes_nuevo.csv"
    )

    return clientes

In [4]:
@task
def guardar_clientes(clientes):

    logger = get_run_logger()

    logger.info("Guardando clientes...")

    clientes.write_csv(
        DATA_PATH / "clientes_transformado.csv"
    )

    logger.info("Proceso finalizado.")

In [5]:
@task
def transformar_clientes(clientes):

    logger = get_run_logger()

    logger.info("Transformando clientes...")

    clientes = clientes.with_columns(

        pl.when(pl.col("edad") < 30)
        .then(pl.lit("Joven"))

        .when(pl.col("edad") < 60)
        .then(pl.lit("Adulto"))

        .otherwise(pl.lit("Senior"))

        .alias("grupo_edad")

    )

    return clientes

In [64]:
@flow(name="Pipeline Clientes")
def pipeline_clientes():

    clientes = leer_clientes()
    clientes = id_duplicados(clientes, "id_cliente")
    clientes = igualar_campos(clientes, "ciudad")

    clientes = transformar_clientes(clientes)

    guardar_clientes(clientes)


In [65]:
pipeline_clientes()

17:55:21.816 | INFO    | Flow run 'emerald-flounder' - Beginning flow run 'emerald-flounder' for flow 'Pipeline Clientes'

17:55:21.818 | INFO    | Flow run 'emerald-flounder' - View at http://127.0.0.1:4200/runs/flow-run/c8a08980-4477-4a97-8146-3addbca9791a

17:55:21.822 | INFO    | Task run 'leer_clientes-185' - Leyendo clientes...

17:55:21.826 | INFO    | Task run 'leer_clientes-185' - Finished in state Completed()

17:55:21.831 | INFO    | Task run 'id_duplicados-12a' - Comprobando duplicados en la columna id_cliente

17:55:21.835 | INFO    | Task run 'id_duplicados-12a' - Se han encontrado 1 IDs duplicados

... (10 outputs left out)

17:55:21.865 | INFO    | Task run 'guardar_clientes-94a' - Guardando clientes...

17:55:21.869 | INFO    | Task run 'guardar_clientes-94a' - Proceso finalizado.

17:55:21.871 | INFO    | Task run 'guardar_clientes-94a' - Finished in state Completed()

17:55:22.822 | INFO    | Flow run 'emerald-flounder' - Finished in state Completed()

# Productos

In [33]:
@task
def leer_productos():

    logger = get_run_logger()

    logger.info("Leyendo productos...")

    productos = pl.read_csv(
        DATA_PATH / "productos_nuevo.csv"
    )

    return productos

In [34]:
@task
def eliminar_registro_por_id(df, columna_id, id_eliminar):
    logger = get_run_logger()

    logger.info(f"Eliminando registros por ID de la columna {columna_id}")

    ids = id_eliminar.get_column(columna_id)

    df_limpio = df.filter(
        ~pl.col(columna_id).is_in(ids)
    )

    logger.info(f"Registros eliminados: {df.height - df_limpio.height}")

    return df_limpio

In [35]:

@task
def id_duplicados(df, columna):
    logger = get_run_logger()
    logger.info(f"Comprobando duplicados en la columna {columna}")

    duplicados = df.filter(
        pl.col(columna).is_duplicated()
    ).select(columna).unique()

    if duplicados.is_empty():
        logger.info("No se han encontrado duplicados")
        return df

    logger.info(f"Se han encontrado {duplicados.height} IDs duplicados")

    return eliminar_registro_por_id(
        df,
        columna,
        duplicados
    )


In [36]:
@task
def igualar_campos(df,columna):
    logger = get_run_logger()
    logger.info(f"Igualando campos de la columna {columna}")
    df_limpio = df.with_columns(
        pl.col(columna).str.to_lowercase()
    )
    logger.info(f"Campos iguales en la columna {columna}")
    return df_limpio

In [37]:
# En este caso ya se que tengo un valor negativo, pero debe ser un valor absoluto, por lo que lo transformo a positivo
@task
def precio_positivo(df,columna):
    logger = get_run_logger()
    logger.info(f"Transformando a positivo la columna {columna}")
    df_limpio = df.with_columns(
        pl.col(columna).abs()
    )
    logger.info(f"Columna {columna} transformada a positivo")
    return df_limpio

In [38]:
@task
def calcular_margen_beneficio(productos):
    logger = get_run_logger()
    logger.info("Calculando el margen de beneficio de los productos")

    productos = productos.with_columns(
        pl.when(pl.col("precio") != 0)
        .then(
            ((pl.col("precio") - pl.col("coste")) / pl.col("precio")) * 100
        )
        .otherwise(None)
        .alias("margen_beneficio")
    )

    return productos

In [39]:
@task
def transformar_productos(productos):

    logger = get_run_logger()

    logger.info("Transformando productos...")

    productos = productos.with_columns(

        pl.when(pl.col("margen_beneficio") < 30)
        .then(pl.lit("Poco rentable"))

        .when(
            (pl.col("margen_beneficio") > 30) &
            (pl.col("margen_beneficio") < 70)
        )
        .then(pl.lit("Rentable"))

        .otherwise(pl.lit("Muy rentable"))

        .alias("grupo_beneficio")

    )

    return productos

In [40]:
@task
def guardar_productos(productos):

    logger = get_run_logger()

    logger.info("Guardando productos...")

    productos.write_csv(
        DATA_PATH / "productos_transformado.csv"
    )

    logger.info("Proceso finalizado.")

In [ ]:
@flow(name="Pipeline Productos")
def pipeline_productos():

    productos = leer_productos()

    productos = id_duplicados(productos, "id_producto")

    productos = igualar_campos(productos, "categoria")
    productos = igualar_campos(productos, "marca")

    productos = precio_positivo(productos, "precio")

    productos = calcular_margen_beneficio(productos)

    productos = transformar_productos(productos)

    guardar_productos(productos)

In [42]:
pipeline_productos()

17:22:52.247 | INFO    | Flow run 'acrid-sunfish' - Beginning flow run 'acrid-sunfish' for flow 'Pipeline Productos'

17:22:52.251 | INFO    | Flow run 'acrid-sunfish' - View at http://127.0.0.1:4200/runs/flow-run/10dd72a6-9769-4976-9036-f9272d28f703

17:22:52.258 | INFO    | Task run 'leer_productos-1ea' - Leyendo productos...

17:22:52.267 | INFO    | Task run 'leer_productos-1ea' - Finished in state Completed()

17:22:52.280 | INFO    | Task run 'id_duplicados-825' - Comprobando duplicados en la columna id_producto

17:22:52.288 | INFO    | Task run 'id_duplicados-825' - Se han encontrado 1 IDs duplicados

... (18 outputs left out)

17:22:52.429 | INFO    | Task run 'guardar_productos-97b' - Guardando productos...

17:22:52.472 | INFO    | Task run 'guardar_productos-97b' - Proceso finalizado.

17:22:52.479 | INFO    | Task run 'guardar_productos-97b' - Finished in state Completed()

17:22:53.254 | INFO    | Flow run 'acrid-sunfish' - Finished in state Completed()

# Ventas

In [57]:
@task
def leer_ventas():

    logger = get_run_logger()

    logger.info("Leyendo ventas...")

    ventas = pl.read_csv(
        DATA_PATH / "ventas_nuevo.csv"
    )

    return ventas

In [58]:
@task
def guardar_ventas(ventas):

    logger = get_run_logger()

    logger.info("Guardando ventas...")

    ventas.write_csv(
        DATA_PATH / "ventas_transformado.csv"
    )

    logger.info("Proceso finalizado.")

In [59]:
@task
def ventas_semanas (ventas):
    logger = get_run_logger()
    logger.info("Calculando el dia de la semana de las ventas...")
    ventas = ventas.with_columns(
        pl.col("fecha").str.to_datetime(format="%Y-%m-%d %H:%M:%S").dt.date().alias("fecha")
    )
    ventas = ventas.with_columns(
      pl.col("fecha").dt.weekday()
        .alias("dia_semana")
    )
    return ventas

In [60]:
@task
def transformar_productos(ventas):

    logger = get_run_logger()

    logger.info("Transformando ventas...")

    ventas = ventas.with_columns(

        pl.when(pl.col("dia_semana") > 5)
        .then(pl.lit("Fin de semana"))

        .otherwise(pl.lit("Semana laboral"))

        .alias("grupo_dia_laboral")

    )

    return ventas

In [61]:
@flow(name="Pipeline ventas")
def pipeline_ventas():

    ventas = leer_ventas()

    ventas = id_duplicados(ventas, "id_venta")

    ventas = igualar_campos(ventas, "metodo_pago")

    ventas = ventas_semanas (ventas)
    ventas = transformar_productos(ventas)

    guardar_ventas(ventas)

In [62]:
pipeline_ventas()

17:34:49.137 | INFO    | Flow run 'yellow-beagle' - Beginning flow run 'yellow-beagle' for flow 'Pipeline ventas'

17:34:49.140 | INFO    | Flow run 'yellow-beagle' - View at http://127.0.0.1:4200/runs/flow-run/5ca68024-ff22-4054-8b2f-b651c4e6f27b

17:34:49.146 | INFO    | Task run 'leer_ventas-32b' - Leyendo ventas...

17:34:49.155 | INFO    | Task run 'leer_ventas-32b' - Finished in state Completed()

17:34:49.171 | INFO    | Task run 'id_duplicados-c57' - Comprobando duplicados en la columna id_venta

17:34:49.176 | INFO    | Task run 'id_duplicados-c57' - Se han encontrado 1 IDs duplicados

... (12 outputs left out)

17:34:49.309 | INFO    | Task run 'guardar_ventas-398' - Guardando ventas...

17:34:49.317 | INFO    | Task run 'guardar_ventas-398' - Proceso finalizado.

17:34:49.325 | INFO    | Task run 'guardar_ventas-398' - Finished in state Completed()

17:34:50.148 | INFO    | Flow run 'yellow-beagle' - Finished in state Completed()